# CryptoPunks floor since 2017, and the PNKSTR premium

A quick start for this dataset: load both files, plot the daily floor on a log scale, and plot the PunkStrategy premium against its own recent range.

The floor is rebuilt from the CryptoPunks marketplace contract's own events. Method: https://punkprice.com/about. Live chart and API: https://punkprice.com/data

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) or glob.glob(f"../upload/{name}")
    return hits[0]

floor = pd.read_csv(find("cryptopunks-daily-floor.csv"), parse_dates=["date"])
premium = pd.read_csv(find("pnkstr-daily-premium.csv"), parse_dates=["date"])
print(f"floor:   {len(floor):,} days, {floor.date.min():%Y-%m-%d} to {floor.date.max():%Y-%m-%d}")
print(f"premium: {len(premium):,} days, {premium.date.min():%Y-%m-%d} to {premium.date.max():%Y-%m-%d}")
floor.tail()

## The floor, every day since June 2017

`floor_eth` is the lowest public ask open at 23:59:59 UTC. A log scale is the only way to see 2017 and 2021 on one chart.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(floor.date, floor.floor_eth, lw=1.2, color="#22b8e6")
ax.set_yscale("log")
ax.set_ylabel("floor (ETH, log scale)")
ax.set_title("CryptoPunks daily floor")
ax.grid(alpha=0.3)
plt.show()

## Yearly summary

How the floor moved year by year, and how often the order book changed (`source == "onchain-orderbook"`).

In [ ]:
yearly = floor.assign(year=floor.date.dt.year).groupby("year").agg(
    low_eth=("floor_eth", "min"),
    high_eth=("floor_eth", "max"),
    year_end_eth=("floor_eth", "last"),
    active_days=("source", lambda s: (s == "onchain-orderbook").sum()),
    avg_open_asks=("active_asks", "mean"),
).round(2)
yearly

## The PNKSTR premium

`premium_x` is PunkStrategy's market cap divided by its treasury's value (punks held at the floor, plus unallocated ETH). 1.0 means the token trades at exactly its treasury.

The bands are a rolling 90-day mean with 1 and 2 standard deviations, computed on the log of the premium. They describe the recent past, nothing more.

In [ ]:
p = premium.set_index("date").premium_x
logp = np.log(p)
mean = logp.rolling(90, min_periods=30).mean()
sd = logp.rolling(90, min_periods=30).std()

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(p.index, np.exp(mean - 2 * sd), np.exp(mean + 2 * sd), color="#22b8e6", alpha=0.10, label="2 sd")
ax.fill_between(p.index, np.exp(mean - sd), np.exp(mean + sd), color="#22b8e6", alpha=0.20, label="1 sd")
ax.plot(p.index, np.exp(mean), color="#22b8e6", lw=1, ls="--", label="90-day mean")
ax.plot(p.index, p, color="black", lw=1.2, label="premium")
ax.set_yscale("log")
ax.set_ylabel("market cap / treasury value (log)")
ax.set_title("PNKSTR premium to its treasury")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

z = (logp.iloc[-1] - mean.iloc[-1]) / sd.iloc[-1]
print(f"latest premium {p.iloc[-1]:.2f}x, {z:+.1f} sd from its 90-day mean")

## Cite

PunkPrice (2026). CryptoPunks daily floor price, reconstructed from the CryptoPunks marketplace contract. https://punkprice.com/data

Licence: CC BY 4.0. Source and verification script: https://github.com/ZonnetjeAXIOM/cryptopunks-floor